In [ ]:
# ==============================================================================
# S&P 500 STAT-ARB PIPELINE: ROBUST 2-STAGE ARCHITECTURE (HIST-GBDT EDITION)
# ==============================================================================
import os
import gc
import pickle
import warnings
from typing import Dict, List, Tuple, Optional
from collections import deque
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import chi2, t as student_t
from scipy.optimize import minimize as scipy_minimize
import vectorbt as vbt
from joblib import Parallel, delayed
from sklearn.ensemble import HistGradientBoostingClassifier

warnings.filterwarnings("ignore")
pd.set_option('display.max_rows', 250)
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_id = "model_A_gbdt_terminal"

# ==============================================================================
# 1. INSTITUTIONAL PARAMETER CONFIGURATION
# ==============================================================================
run_params = {
    "price_csv": "data/universe_daily_train.csv",
    "oos_price_csv": "data/universe_daily_val.csv",
    "constituents_path": "data/constituents.csv",
    "state_persistence_file": f"artifacts/{model_id}/stat_arb_state_{timestamp}.pkl",

    "lookback_days": 504,             
    "min_history_days": 100,          
    
    "weight_drift_threshold": 0.05,     
    "screening_freq_days": 20,          
    "quarantine_days": 42,             
    
    "kalman_delta": 1e-4,              
    "kalman_obs_noise": 1e-3,          
    "kalman_z_window": 42,             
    "max_hurst_exponent": 0.45,        
    
    "min_correlation": 0.40,           
    "p_value_threshold": 0.20,         
    
    "min_half_life": 2.0,             
    "max_half_life": 45.0,            
    "tau_hl_penalty": 5.0,             
    
    "exit_z_spread": 0.10,            
    "stop_loss_z": 3.00,               
    "bocpd_cp_threshold": 0.35,        
    
    # HIST-GBDT PARAMS
    "gbdt_max_depth": 3,
    "gbdt_buffer_size": 250,           # Rolling window of trade resolutions
    "gbdt_min_samples": 50,            # Wait for 50 trades before using model
    "min_conviction": 0.40,
    "promote_top_n": 30,               
    "ghost_truncation_quantile": 0.50, 
    
    # SLSQP RISK BOUNDS
    "max_net_dollar_exposure": 0.05,   
    "max_sector_exposure": 0.10,       
    "max_gross_exposure": 1.8,        
    "max_sleeve_weight": 0.25,        

    "max_adv_participation": 0.015,    
    "initial_capital": 250_000.0,
    "exec_fee": 0.00015,              
    "borrow_bps": 50.0
}

# ==============================================================================
# 2. STATE PERSISTENCE ENGINE
# ==============================================================================
class StatePersistenceEngine:
    @staticmethod
    def save_checkpoint(filepath: str, state_dict: dict):
        try:
            os.makedirs(os.path.dirname(filepath), exist_ok=True)
            with open(filepath, "wb") as f:
                pickle.dump(state_dict, f)
        except Exception:
            pass

    @staticmethod
    def load_checkpoint(filepath: str) -> Optional[dict]:
        if not os.path.exists(filepath): return None
        try:
            with open(filepath, "rb") as f:
                return pickle.load(f)
        except Exception:
            return None

# ==============================================================================
# 3. MATH & KALMAN UTILITIES
# ==============================================================================
def calculate_half_life_fast(spread: np.ndarray) -> float:
    if len(spread) < 30: return 25.0
    s_lag, s_diff = spread[:-1], spread[1:] - spread[:-1]
    var_x = np.sum((s_lag - np.mean(s_lag))**2)
    if var_x < 1e-12: return 25.0
    gamma = np.sum((s_lag - np.mean(s_lag)) * (s_diff - np.mean(s_diff))) / var_x
    if gamma <= -1.0: return 1.0
    if gamma >= 0.0: return np.inf
    return float(-0.6931471805599453 / np.log(1.0 + gamma))

def compute_hurst_exponent_fast(time_series: np.ndarray, max_lag: int = 20) -> float:
    N = len(time_series)
    if N < max_lag * 2: return 0.50
    lags = np.arange(2, max_lag)
    tau = [np.std(time_series[lag:] - time_series[:-lag]) for lag in lags]
    slope = np.polyfit(np.log(lags), np.log(np.maximum(tau, 1e-8)), 1)[0]
    return float(slope * 2.0)

def compute_sortino_ratio(returns: pd.Series, target_return: float = 0.0) -> float:
    if len(returns) < 5 or returns.std() < 1e-8: return 0.0
    downside_std = np.sqrt(np.mean(returns[returns < target_return]**2)) * np.sqrt(252.0)
    return float((returns.mean() * 252.0) / max(downside_std, 1e-6))

class FastKalmanFilter:
    def __init__(self, delta: float = 1e-4, obs_noise: float = 1e-3, z_window: int = 21):
        self.delta, self.obs_noise, self.z_window = delta, obs_noise, z_window
        self.Q = (delta / (1.0 - delta)) * np.eye(2, dtype=np.float64)

    def filter_series(self, y: np.ndarray, x: np.ndarray) -> Dict[str, np.ndarray]:
        N = len(y)
        state_mean, state_cov = np.zeros(2, dtype=np.float64), np.eye(2, dtype=np.float64)
        betas, alphas, v_arr = np.empty(N), np.empty(N), np.empty(N)
        H = np.ones((2, 1), dtype=np.float64)
        for t in range(N):
            H[1, 0] = x[t]
            pred_cov = state_cov + self.Q
            v = y[t] - (H.T @ state_mean).item()
            F = (H.T @ pred_cov @ H).item() + self.obs_noise
            K = (pred_cov @ H) / F
            state_mean += (K.flatten() * v)
            state_cov = (np.eye(2) - K @ H.T) @ pred_cov
            alphas[t], betas[t], v_arr[t] = state_mean[0], state_mean[1], v

        F_emp = pd.Series(v_arr).rolling(self.z_window, min_periods=5).std().bfill().values**2
        e_t = v_arr / np.sqrt(np.maximum(F_emp, 1e-4))
        return {"Beta": betas, "Innovation": v_arr, "Innovation_Var": F_emp, "Innovation_Z": e_t, "Final_State_Mean": state_mean, "Final_State_Cov": state_cov}

    def step_update(self, state_mean: np.ndarray, state_cov: np.ndarray, y_val: float, x_val: float):
        H = np.array([1.0, x_val], dtype=np.float64)
        pred_cov = state_cov + self.Q
        v_t = y_val - float(H @ state_mean)
        F_t = float(H @ pred_cov @ H.T) + self.obs_noise
        K = (pred_cov @ H) / F_t
        return state_mean + K * v_t, (np.eye(2) - np.outer(K, H)) @ pred_cov, v_t, F_t

class FastBOCPDEngine:
    def __init__(self, hazard_rate: float = 1.0 / 200.0, max_K: int = 64):
        self.H, self.max_K = hazard_rate, max_K

    def eval_change_point_prob(self, innovations: np.ndarray) -> float:
        T_len = min(len(innovations), self.max_K)
        if T_len < 5: return 0.0
        recent_std, hist_std = np.std(innovations[-5:]), np.std(innovations)
        return float(min(1.0, (recent_std / (hist_std + 1e-5)) - 1.0)) if recent_std > hist_std else 0.0

# ==============================================================================
# 4. HIST-GBDT ROLLING CONVICTION ENGINE (TERMINAL RESOLUTION)
# ==============================================================================
class HistGBDTConvictionEngine:
    def __init__(self, max_depth: int = 3, buffer_size: int = 250, min_samples: int = 50):
        self.model = HistGradientBoostingClassifier(
            max_depth=max_depth,
            learning_rate=0.05,
            max_iter=100,
            early_stopping=False,
            random_state=42
        )
        self.buffer_X = deque(maxlen=buffer_size)
        self.buffer_y = deque(maxlen=buffer_size)
        self.min_samples = min_samples
        self.is_trained = False
        self.needs_retrain = False

    def predict_conviction(self, context_vector: np.ndarray) -> float:
        if not self.is_trained: return 0.50 
        X = np.array([context_vector])
        return float(np.clip(self.model.predict_proba(X)[0, 1], 0.10, 0.95))

    def update_terminal(self, context_vector: np.ndarray, entry_z: float, exit_z: float):
        """Buffers resolved trades. 1 if convergence, 0 if divergence/stop-out."""
        target = 1 if abs(exit_z) < abs(entry_z) * 0.5 else 0
        self.buffer_X.append(context_vector)
        self.buffer_y.append(target)
        self.needs_retrain = True

    def fit_daily(self):
        """Called once per day to refit the rolling tree ensemble if updates occurred."""
        if self.needs_retrain and len(self.buffer_X) >= self.min_samples:
            if len(set(self.buffer_y)) > 1: # Require both classes to fit
                X_arr = np.array(self.buffer_X)
                y_arr = np.array(self.buffer_y)
                self.model.fit(X_arr, y_arr)
                self.is_trained = True
        self.needs_retrain = False

class UniverseScreener:
    def __init__(self, constituents_path: str):
        self.constituents_path = constituents_path

    def get_sectors(self, tickers: List[str]) -> pd.Series:
        if os.path.exists(self.constituents_path):
            const = pd.read_csv(self.constituents_path)
            sym_col = "Symbol" if "Symbol" in const.columns else "Ticker"
            return pd.Series(tickers, index=tickers).map(dict(zip(const[sym_col], const["GICS Sector"]))).fillna("Unknown")
        return pd.Series("Unknown", index=tickers)

# ==============================================================================
# 5. FAST PAIR COINTEGRATION ENGINE
# ==============================================================================
class PairCointegrationEngine:
    def __init__(self, params: dict):
        self.params = params
        self.kalman_engine = FastKalmanFilter(delta=params.get("kalman_delta", 1e-4), z_window=params.get("kalman_z_window", 21))

    def _screen_single_pair_fast(self, t1: str, t2: str, p1: np.ndarray, p2: np.ndarray) -> Optional[Tuple[str, str]]:
        mask = ~np.isnan(p1) & ~np.isnan(p2)
        if np.sum(mask) < self.params["min_history_days"]: return None
        if coint(p1[mask], p2[mask])[1] <= self.params["p_value_threshold"]: return (t1, t2)
        return None

    def screen_universe_cointegration(self, price_df: pd.DataFrame, sectors: pd.Series, parallel) -> List[Tuple[str, str]]:
        valid_df = price_df.ffill().dropna(axis=1)
        valid_tickers = list(valid_df.columns)
        if len(valid_tickers) < 2:
            return []
        log_prices = np.log(valid_df.values)
        corr_matrix = np.corrcoef(np.diff(log_prices, axis=0), rowvar=False)
        sector_arr = sectors.reindex(valid_tickers).values
        sec_match = sector_arr[:, None] == sector_arr[None, :]
        mask = np.triu(sec_match & (corr_matrix >= self.params["min_correlation"]), k=1)
        candidates = [(valid_tickers[i], valid_tickers[j], log_prices[:, i], log_prices[:, j]) for i, j in zip(*np.where(mask))]
        results = parallel(delayed(self._screen_single_pair_fast)(*c) for c in candidates)
        return [r for r in results if r is not None]

    def _test_pair(self, t1: str, t2: str, price_df: pd.DataFrame, vix: float, vol_ratio: float, gbdt_engine: HistGBDTConvictionEngine, active_data: dict = None):
        s1, s2 = price_df[t1].dropna(), price_df[t2].dropna()
        idx = s1.index.intersection(s2.index)
        if len(idx) < self.params["min_history_days"]: return None

        is_active = active_data is not None
        kf_out = self.kalman_engine.filter_series(np.log(s1.loc[idx].values), np.log(s2.loc[idx].values))
        beta, e_t, F_t = kf_out["Beta"][-1], kf_out["Innovation_Z"][-1], kf_out["Innovation_Var"][-1]
        
        static_spread = np.log(s1.loc[idx].values) - (beta * np.log(s2.loc[idx].values))
        
        hurst_val = compute_hurst_exponent_fast(static_spread[-126:])
        if not is_active and hurst_val >= self.params["max_hurst_exponent"]: return None

        hl = float(np.clip(calculate_half_life_fast(static_spread), self.params["min_half_life"], self.params["max_half_life"]))
        if not is_active and hl == self.params["max_half_life"]: return None

        ctx = np.array([vol_ratio, vix, abs(e_t), hl, hurst_val])
        conviction = gbdt_engine.predict_conviction(ctx)

        return {
            "Asset_A": t1, "Asset_B": t2, "Beta": beta, "Half_Life_Days": hl, "Z_Spread": e_t,
            "Conviction_Score": conviction, "Context_Vector": ctx, "Spread_Vol": np.sqrt(F_t), 
            "Days_Held": active_data.get("Days_Held", 0) + 5 if is_active else 0,
            "Is_Active": is_active, "Entry_Z": active_data["Entry_Z"] if is_active else e_t,
            "Ghost_Returns": pd.Series(static_spread).diff().fillna(0).values[-63:],
            "State_Mean": kf_out["Final_State_Mean"], "State_Cov": kf_out["Final_State_Cov"],
            "Recent_V_Buffer": deque(kf_out["Innovation"][-self.kalman_engine.z_window:], maxlen=42)
        }

    def find_pairs(self, price_df: pd.DataFrame, sectors: pd.Series, active_pairs: dict, cached_keys: List[Tuple[str, str]], vix: float, vol_ratio: float, gbdt_engine, parallel):
        tests = [(k[0], k[1], act) for k, act in active_pairs.items()]
        seen = set(active_pairs.keys())
        tests += [(t1, t2, None) for t1, t2 in cached_keys if (t1, t2) not in seen and (t2, t1) not in seen]
        results = parallel(delayed(self._test_pair)(t1, t2, price_df, vix, vol_ratio, gbdt_engine, act) for t1, t2, act in tests)
        return pd.DataFrame([r for r in results if r is not None])

# ==============================================================================
# 6. CONVICTION-BOUNDED SLSQP OPTIMIZER (STRICT NEUTRALITY)
# ==============================================================================
class ConvexPortfolioOptimizer:
    def __init__(self, params: dict):
        self.params = params

    @staticmethod
    def calculate_max_weight_drift(current_sleeve_weights: Dict[Tuple[str, str], float], target_sleeve_weights: Dict[Tuple[str, str], float]) -> float:
        all_keys = set(current_sleeve_weights.keys()).union(target_sleeve_weights.keys())
        if not all_keys: return 0.0
        return float(max([abs(current_sleeve_weights.get(pk, 0.0) - target_sleeve_weights.get(pk, 0.0)) for pk in all_keys]))

    def allocate(self, live_df: pd.DataFrame, sectors: pd.Series, current_weights: dict):
        if live_df.empty: return {}, {}, {}

        N = len(live_df)
        pair_keys = [(r["Asset_A"], r["Asset_B"]) for _, r in live_df.iterrows()]
        
        returns_mat = np.column_stack([live_df.iloc[i]["Ghost_Returns"] for i in range(N)])
        cov_matrix = np.cov(returns_mat, rowvar=False) * 252.0 + np.eye(N) * 1e-5
        
        pair_betas, pair_zs = live_df["Beta"].values, live_df["Z_Spread"].values
        spread_vols, convictions = live_df["Spread_Vol"].values, live_df["Conviction_Score"].values

        w_prev = np.array([current_weights.get(pk, 0.0) for pk in pair_keys])
        turnover_bps = (self.params["exec_fee"] * 2.0) + 0.0005 

        dollar_mults = np.where(pair_zs > 0, pair_betas - 1.0, 1.0 - pair_betas)
        pair_secs = np.array([sectors.get(live_df.iloc[i]["Asset_A"], "Unknown") for i in range(N)])
        unique_secs = np.unique(pair_secs)

        def objective(w):
            risk = np.dot(w.T, np.dot(cov_matrix, w))
            turnover = np.sum(np.abs(w - w_prev)) * turnover_bps
            return risk + turnover

        target_gross = min(self.params["max_gross_exposure"], N * self.params["max_sleeve_weight"])
        
        max_net = self.params["max_net_dollar_exposure"]
        max_sec = self.params["max_sector_exposure"]
        
        constraints = [{'type': 'eq', 'fun': lambda w: target_gross - np.sum(w)}]
        constraints.append({'type': 'ineq', 'fun': lambda w: max_net - np.abs(np.sum(w * dollar_mults))})
        for s in unique_secs:
            constraints.append({'type': 'ineq', 'fun': lambda w, sec=s: max_sec - np.abs(np.sum(w[pair_secs == sec] * dollar_mults[pair_secs == sec]))})

        base_cap = self.params["max_sleeve_weight"]
        bounds = [(0.0, base_cap * (0.50 + 0.50 * float(convictions[i]))) for i in range(N)]
        
        init_w = (1.0 / np.maximum(spread_vols, 1e-4))
        init_w = np.minimum((init_w / np.sum(init_w)) * target_gross, [b[1] for b in bounds])

        res = scipy_minimize(objective, init_w, method='SLSQP', bounds=bounds, constraints=constraints)
        best_w = res.x if res.success else init_w
        best_w[best_w < 0.01] = 0.0

        allocs, memory = {}, {}
        for i, pk in enumerate(pair_keys):
            if best_w[i] > 0:
                row = live_df.iloc[i]
                w_a = -best_w[i] if row["Z_Spread"] > 0 else best_w[i]
                w_b = best_w[i] * row["Beta"] if row["Z_Spread"] > 0 else -best_w[i] * row["Beta"]
                allocs[pk] = {row["Asset_A"]: w_a, row["Asset_B"]: w_b, "Sleeve_Weight": best_w[i]}
                memory[pk] = {
                    "Beta": row["Beta"], "Entry_Z": row["Entry_Z"] if row["Is_Active"] else row["Z_Spread"],
                    "Conviction_Score": row["Conviction_Score"], "Context_Vector": row["Context_Vector"],
                    "Half_Life_Days": row["Half_Life_Days"], "Days_Held": row.get("Days_Held", 0),
                    "State_Mean": row["State_Mean"], "State_Cov": row["State_Cov"], "Recent_V_Buffer": row["Recent_V_Buffer"]
                }

        metrics = {"Portfolio_Variance": np.dot(best_w.T, np.dot(cov_matrix, best_w)), "Turnover_Cost": np.sum(np.abs(best_w - w_prev)) * turnover_bps, "Effective_Gross": np.sum(best_w)}
        return allocs, memory, metrics

# ==============================================================================
# 7. BACKTEST ENGINES & DIAGNOSTICS
# ==============================================================================
class CommitteeTearsheetEngine:
    def __init__(self, params: dict):
        self.initial_capital = params.get("initial_capital", 250_000.0)
        self.exec_fee = params.get("exec_fee", 0.00015)
        self.daily_borrow_rate = (params.get("borrow_bps", 50.0) / 10000.0) / 252.0

    def generate_tearsheets(self, prices_df: pd.DataFrame, pair_weight_matrices: Dict[Tuple[str, str], pd.DataFrame]) -> pd.DataFrame:
        results = []
        for pair_key, w_df in pair_weight_matrices.items():
            a, b = pair_key
            active_dates = w_df.dropna(how='all').index
            if len(active_dates) == 0: continue
            start_date = active_dates[0]
            p_oos = prices_df[[a, b]].loc[start_date:]
            w_target = w_df.loc[start_date:].copy()
            w_target.iloc[-1] = 0.0 
            
            pf_gross = vbt.Portfolio.from_orders(
                close=p_oos, size=w_target, size_type='targetpercent',
                group_by=True, cash_sharing=True, init_cash=self.initial_capital, fees=self.exec_fee
            )
            asset_values = pf_gross.asset_value(group_by=False)
            short_exposure = asset_values.where(asset_values < 0, 0).abs()
            daily_short_cost = short_exposure.sum(axis=1) * self.daily_borrow_rate
            net_val = pf_gross.value() - daily_short_cost.cumsum()
            net_ret = net_val.pct_change().fillna(0)
            
            tot_ret = (net_val.iloc[-1] / self.initial_capital) - 1.0
            mean_ret, std_ret = net_ret.mean(), net_ret.std()
            down_std = np.sqrt((net_ret[net_ret < 0] ** 2).mean())
            sharpe = (mean_ret / std_ret) * np.sqrt(252) if std_ret > 0 else 0.0
            sortino = (mean_ret / down_std) * np.sqrt(252) if down_std > 0 else 0.0
            max_dd = (1 - net_val / net_val.cummax()).max() if not net_val.empty else 0.0
            trades = pf_gross.trades.count()
            win_rate = (pf_gross.trades.winning.count() / trades) if trades > 0 else 0.0
            
            results.append({
                "Pair_Legs": f"{a} / {b}", "Total_Return_[%]": tot_ret * 100,
                "Sharpe_Ratio": sharpe, "Sortino_Ratio": sortino,
                "Max_DD_[%]": max_dd * 100, "Win_Rate_[%]": win_rate * 100, "Total_Trades": trades
            })
            del pf_gross
            gc.collect()

        if not results: return pd.DataFrame()
        return pd.DataFrame(results).sort_values("Sharpe_Ratio", ascending=False).reset_index(drop=True)

class VectorbtBacktestEngine:
    def __init__(self, params: dict):
        self.initial_capital = params.get("initial_capital", 250_000.0)
        self.exec_fee = params.get("exec_fee", 0.00015)
        self.daily_borrow_rate = (params.get("borrow_bps", 50.0) / 10000.0) / 252.0

    def run_backtest(self, close_prices_df: pd.DataFrame, open_prices_df: pd.DataFrame, weights_df: pd.DataFrame) -> Tuple[vbt.Portfolio, pd.Series, pd.Series]:
        tradable_assets = weights_df.columns.intersection(close_prices_df.columns)
        p_close_oos, p_open_oos = close_prices_df[tradable_assets].ffill(), open_prices_df[tradable_assets].ffill()
        sparse_weights = weights_df[tradable_assets]

        pf_gross = vbt.Portfolio.from_orders(
            close=p_close_oos, price=p_open_oos, size=sparse_weights, size_type='targetpercent',
            group_by=True, cash_sharing=True, init_cash=self.initial_capital, fees=self.exec_fee
        )
        
        asset_values = pf_gross.asset_value(group_by=False)
        short_exposure = asset_values.where(asset_values < 0, 0).abs()
        daily_short_cost = short_exposure.sum(axis=1) * self.daily_borrow_rate
        net_portfolio_value = pf_gross.value() - daily_short_cost.cumsum()
        
        total_abs_asset_val = asset_values.abs().sum(axis=1)
        true_gross_exposure = (total_abs_asset_val / net_portfolio_value.replace(0.0, np.nan)).fillna(0.0)
        return pf_gross, net_portfolio_value, true_gross_exposure

class MarketNeutralityDiagnosticEngine:
    def __init__(self, max_beta: float = 0.03, max_r2: float = 0.01):
        self.max_beta, self.max_r2 = max_beta, max_r2

    def analyze(self, strategy_returns: pd.Series, benchmark_returns: pd.Series) -> Tuple[pd.DataFrame, dict]:
        aligned_df = pd.concat([strategy_returns, benchmark_returns], axis=1).dropna()
        aligned_df.columns = ["Strategy", "Benchmark"]
        r_p, r_m = aligned_df["Strategy"], aligned_df["Benchmark"]
        
        capm_model = sm.OLS(r_p, sm.add_constant(r_m)).fit()
        alpha_ann = capm_model.params.get("const", 0.0) * 252.0
        beta_m = capm_model.params.get("Benchmark", 0.0)
        p_val_beta = capm_model.pvalues.get("Benchmark", 1.0)
        t_stat_beta = capm_model.tvalues.get("Benchmark", 0.0)
        r2_capm = capm_model.rsquared

        bull_mask, bear_mask = r_m > 0, r_m < 0
        beta_bull = sm.OLS(r_p[bull_mask], sm.add_constant(r_m[bull_mask])).fit().params.get("Benchmark", 0.0) if bull_mask.sum() > 10 else 0.0
        beta_bear = sm.OLS(r_p[bear_mask], sm.add_constant(r_m[bear_mask])).fit().params.get("Benchmark", 0.0) if bear_mask.sum() > 10 else 0.0

        metrics = [
            {"Metric": "Annualized Alpha", "Value": f"{alpha_ann:.2%}", "Threshold": "N/A", "Status": "INFO"},
            {"Metric": "Market Beta (β_m)", "Value": f"{beta_m:.4f}", "Threshold": f"< |{self.max_beta}|", "Status": "PASS" if abs(beta_m) <= self.max_beta else "FAIL"},
            {"Metric": "Beta p-value", "Value": f"{p_val_beta:.4f}", "Threshold": ">= 0.05", "Status": "PASS" if p_val_beta >= 0.05 else "FAIL"},
            {"Metric": "Beta t-statistic", "Value": f"{t_stat_beta:.4f}", "Threshold": "< |1.96|", "Status": "PASS" if abs(t_stat_beta) < 1.96 else "FAIL"},
            {"Metric": "Variance Explained (R²)", "Value": f"{r2_capm:.2%}", "Threshold": f"< {self.max_r2:.1%}", "Status": "PASS" if r2_capm <= self.max_r2 else "FAIL"},
            {"Metric": "Bull Market Beta (R_m > 0)", "Value": f"{beta_bull:.4f}", "Threshold": f"< |{self.max_beta}|", "Status": "PASS" if abs(beta_bull) <= self.max_beta else "FAIL"},
            {"Metric": "Bear Market Beta (R_m < 0)", "Value": f"{beta_bear:.4f}", "Threshold": f"< |{self.max_beta}|", "Status": "PASS" if abs(beta_bear) <= self.max_beta else "FAIL"}
        ]
        metrics_df = pd.DataFrame(metrics)
        all_passed = (metrics_df["Status"] != "FAIL").all()
        verdict = {"Market_Neutral": "TRUE" if all_passed else "FALSE", "Core_Failure_Reason": "None" if all_passed else ", ".join(metrics_df[metrics_df["Status"] == "FAIL"]["Metric"].tolist())}
        return metrics_df, verdict

class LiveMOOOrderGenerator:
    def __init__(self, min_order_usd: float = 250.0, max_adv_part: float = 0.015, round_lots: bool = False):
        self.min_order_usd, self.max_adv_part, self.round_lots = min_order_usd, max_adv_part, round_lots

    def generate_blotter(self, target_weights: pd.Series, current_positions: Dict[str, int], latest_prices: pd.Series, rolling_adv_shares: pd.Series, portfolio_nav: float) -> pd.DataFrame:
        blotter = []
        all_tickers = set(target_weights.index).union(current_positions.keys())
        for ticker in all_tickers:
            t_weight, price, adv_shares = float(target_weights.get(ticker, 0.0)), float(latest_prices.get(ticker, np.nan)), float(rolling_adv_shares.get(ticker, 1_000_000.0))
            if np.isnan(price) or price <= 0: continue

            curr_shares = int(current_positions.get(ticker, 0))
            raw_target_shares = int((t_weight * portfolio_nav) / price)
            max_allowed_delta = int(adv_shares * self.max_adv_part)
            desired_delta = raw_target_shares - curr_shares
            
            target_shares = curr_shares + (max_allowed_delta if desired_delta > max_allowed_delta else (-max_allowed_delta if desired_delta < -max_allowed_delta else desired_delta))
            if self.round_lots: target_shares = (target_shares // 100) * 100

            delta_shares = target_shares - curr_shares
            delta_value = delta_shares * price

            if abs(delta_value) < self.min_order_usd and target_shares != 0: continue

            if delta_shares != 0:
                blotter.append({
                    "Ticker": ticker, "Action": "BUY" if delta_shares > 0 else "SELL", "Order_Type": "MOO",
                    "Delta_Shares": abs(delta_shares), "Target_Shares": target_shares, "Current_Shares": curr_shares,
                    "Est_Order_USD": round(abs(delta_value), 2), "Target_Weight_%": round((target_shares * price / portfolio_nav) * 100, 2),
                    "ADV_Participation_%": round((abs(delta_shares) / max(1.0, adv_shares)) * 100, 3), "Price_Ref": round(price, 2)
                })

        df_blotter = pd.DataFrame(blotter)
        if df_blotter.empty: return pd.DataFrame(columns=["Ticker", "Action", "Order_Type", "Delta_Shares", "Target_Shares", "Current_Shares", "Est_Order_USD", "Target_Weight_%", "ADV_Participation_%", "Price_Ref"])
        return df_blotter.sort_values("Est_Order_USD", ascending=False).reset_index(drop=True)

# ==============================================================================
# 8. UNIFIED 2-STAGE ORCHESTRATOR
# ==============================================================================
def run_unified_pipeline(params: dict):
    print("=" * 85, flush=True)
    print("STARTING ROBUST 2-STAGE STAT-ARB ENGINE (TRUNCATED GHOST -> HIST-GBDT SLSQP)", flush=True)
    print("=" * 85, flush=True)
    
    try:
        prices_train = pd.read_csv(params["price_csv"], index_col=0, parse_dates=True)
        prices_val = pd.read_csv(params["oos_price_csv"], index_col=0, parse_dates=True)
        prices_full = pd.concat([prices_train, prices_val]).dropna(axis=1, how="all").ffill()
    except Exception:
        prices_train = pd.read_csv(params["price_csv"], index_col=0, parse_dates=True)
        prices_full = prices_train.ffill()

    prices_full = prices_full[~prices_full.index.duplicated()].sort_index()

    try:
        open_train = pd.read_csv(params["price_csv"].replace("close", "open"), index_col=0, parse_dates=True)
        open_val = pd.read_csv(params["oos_price_csv"].replace("close", "open"), index_col=0, parse_dates=True)
        open_full = pd.concat([open_train, open_val]).dropna(axis=1, how="all").ffill()
        open_full = open_full[~open_full.index.duplicated(keep='first')].sort_index()
    except FileNotFoundError:
        open_full = prices_full.copy()

    screener = UniverseScreener(params["constituents_path"])
    engine_coint = PairCointegrationEngine(params)
    engine_alloc = ConvexPortfolioOptimizer(params)
    
    gbdt_engine = HistGBDTConvictionEngine(
        max_depth=params.get("gbdt_max_depth", 3),
        buffer_size=params.get("gbdt_buffer_size", 250),
        min_samples=params.get("gbdt_min_samples", 50)
    )
    
    state = StatePersistenceEngine.load_checkpoint(params["state_persistence_file"]) or {}
    active_pairs, cooldown_tracker = state.get("active_pairs", {}), state.get("cooldown_tracker", {})
    
    sectors = screener.get_sectors(prices_full.columns.tolist())
    market_ret = prices_full.pct_change().mean(axis=1)
    
    start_idx = min(params["lookback_days"], len(prices_train))
    total_steps = len(prices_full) - start_idx

    master_weights = pd.DataFrame(np.nan, index=prices_full.index, columns=prices_full.columns, dtype=np.float64)
    ghost_weights = pd.DataFrame(np.nan, index=prices_full.index, columns=prices_full.columns, dtype=np.float64)
    promoted_weights = pd.DataFrame(np.nan, index=prices_full.index, columns=prices_full.columns, dtype=np.float64)
    
    pair_weight_matrices, current_sleeve_weights, target_sleeve_weights = {}, {}, {}
    metrics_history, cached_keys = [], []
    days_since_screening = 0

    with Parallel(n_jobs=-1, backend="loky") as parallel:
        for i in range(start_idx, len(prices_full)):
            current_date = prices_full.index[i]
            step_num = i - start_idx + 1
            progress_pct = (step_num / total_steps) * 100.0

            lookback_prices = prices_full.iloc[i - start_idx : i]
            
            vol_ratio = market_ret.iloc[i-5:i].std() / max(market_ret.iloc[i-63:i].std(), 1e-6)
            vix_level = market_ret.iloc[i-5:i].std() * np.sqrt(252) * 100.0

            has_exit = False
            for p in list(active_pairs.keys()):
                y, x = np.log(prices_full[p[0]].iloc[i]), np.log(prices_full[p[1]].iloc[i])
                mem = active_pairs[p]
                
                mean, cov, v_t, F_t = engine_coint.kalman_engine.step_update(mem["State_Mean"], mem["State_Cov"], y, x)
                mem["Recent_V_Buffer"].append(v_t)
                
                e_t = v_t / np.sqrt(max(F_t, 1e-4))
                mem.update({"State_Mean": mean, "State_Cov": cov, "Days_Held": mem["Days_Held"] + 1})

                if abs(e_t) <= params["exit_z_spread"] or abs(e_t) >= params["stop_loss_z"] or mem["Days_Held"] > (mem["Half_Life_Days"] * 2.5):
                    has_exit = True
                    master_weights.loc[current_date, p[0]], master_weights.loc[current_date, p[1]] = 0.0, 0.0
                    if p in pair_weight_matrices:
                        pair_weight_matrices[p].loc[current_date, p[0]] = 0.0
                        pair_weight_matrices[p].loc[current_date, p[1]] = 0.0

                    if abs(e_t) >= params["stop_loss_z"]: cooldown_tracker[p] = params["quarantine_days"]
                    
                    gbdt_engine.update_terminal(mem["Context_Vector"], mem["Entry_Z"], e_t)
                    
                    del active_pairs[p]
                    if p in current_sleeve_weights: del current_sleeve_weights[p]

            # Daily Retrain of GBDT (if updates occurred)
            gbdt_engine.fit_daily()

            for p in list(cooldown_tracker.keys()):
                cooldown_tracker[p] -= 1
                if cooldown_tracker[p] <= 0: del cooldown_tracker[p]

            if days_since_screening >= params["screening_freq_days"] or not cached_keys:
                print(f"\n[Screening Pass @ {current_date.date()} | Progress: {progress_pct:5.1f}% ({step_num}/{total_steps})] SciPy C-Accelerated Search...", flush=True)
                cached_keys = engine_coint.screen_universe_cointegration(lookback_prices, sectors, parallel)
                days_since_screening = 0
            days_since_screening += 1

            ghost_df = engine_coint.find_pairs(lookback_prices, sectors, active_pairs, cached_keys, vix_level, vol_ratio, gbdt_engine, parallel)
            
            promoted_ghosts = pd.DataFrame()
            if not ghost_df.empty:
                ghost_df = ghost_df[~ghost_df.apply(lambda r: (r["Asset_A"], r["Asset_B"]) in cooldown_tracker and not r["Is_Active"], axis=1)]
                ghost_pool = ghost_df[~ghost_df["Is_Active"]]
                
                if not ghost_pool.empty:
                    min_score = max(params["min_conviction"], ghost_pool["Conviction_Score"].quantile(params["ghost_truncation_quantile"]))
                    promoted_ghosts = ghost_pool[ghost_pool["Conviction_Score"] >= min_score]
                    promoted_ghosts = promoted_ghosts.sort_values("Conviction_Score", ascending=False).head(params["promote_top_n"])

                if len(ghost_pool) > 0:
                    g_alloc = 1.0 / len(ghost_pool)
                    for _, r in ghost_pool.iterrows():
                        ghost_weights.loc[current_date, r["Asset_A"]] = -g_alloc if r["Z_Spread"] > 0 else g_alloc
                        ghost_weights.loc[current_date, r["Asset_B"]] = g_alloc * r["Beta"] if r["Z_Spread"] > 0 else -g_alloc * r["Beta"]
                
                if len(promoted_ghosts) > 0:
                    p_alloc = 1.0 / len(promoted_ghosts)
                    for _, r in promoted_ghosts.iterrows():
                        promoted_weights.loc[current_date, r["Asset_A"]] = -p_alloc if r["Z_Spread"] > 0 else p_alloc
                        promoted_weights.loc[current_date, r["Asset_B"]] = p_alloc * r["Beta"] if r["Z_Spread"] > 0 else -p_alloc * r["Beta"]

            current_max_drift = ConvexPortfolioOptimizer.calculate_max_weight_drift(current_sleeve_weights, target_sleeve_weights)
            has_drift_trigger = current_max_drift >= params["weight_drift_threshold"]
            is_initial_day = (i == start_idx)

            should_rebalance = has_exit or len(promoted_ghosts) > 0 or has_drift_trigger or is_initial_day

            if should_rebalance:
                trigger_cause = (
                    "INITIAL_ENTRY" if is_initial_day else
                    ("SIGNAL_EXIT" if has_exit else
                    ("SIGNAL_ENTRY" if not promoted_ghosts.empty else f"WEIGHT_DRIFT ({current_max_drift:.1%})"))
                )

                print(f"\n[Dynamic Rebalance @ {current_date.date()} | Progress: {progress_pct:5.1f}% ({step_num}/{total_steps})] Cause: {trigger_cause} | Vol Ratio: {vol_ratio:.2f} | VIX: {vix_level:.1f}", flush=True)
                
                if ghost_df.empty:
                    print(f"  --> Pipeline Funnel : Ghost=0 | Promoted=0 | Live={len(active_pairs)}", flush=True)
                    metrics_history.append({"Date": current_date, "Ghost_Size": 0, "Promoted_Size": 0, "Live_Size": len(active_pairs), "Opt_Exp_Variance": 0.0, "Effective_Gross": 0.0, "Mean_Innovation_Z": 0.0, "Mean_Conviction": 0.0, "Vol_Ratio": vol_ratio, "rebalanced": 1})
                    continue

                live_candidates = ghost_df[ghost_df["Is_Active"]]
                live_df = pd.concat([live_candidates, promoted_ghosts]).drop_duplicates(subset=["Asset_A", "Asset_B"])
                
                allocs, active_pairs_updates, opt = engine_alloc.allocate(live_df, sectors, current_sleeve_weights)
                
                dropped = set(active_pairs.keys()) - set(allocs.keys())
                for p in dropped:
                    master_weights.loc[current_date, p[0]], master_weights.loc[current_date, p[1]] = 0.0, 0.0
                    if p in pair_weight_matrices:
                        pair_weight_matrices[p].loc[current_date, p[0]] = 0.0
                        pair_weight_matrices[p].loc[current_date, p[1]] = 0.0
                    cooldown_tracker[p] = params["quarantine_days"]

                active_pairs = active_pairs_updates
                current_sleeve_weights = {p: w["Sleeve_Weight"] for p, w in allocs.items()}
                target_sleeve_weights = current_sleeve_weights.copy()

                for p, w in allocs.items():
                    if p not in pair_weight_matrices: pair_weight_matrices[p] = pd.DataFrame(np.nan, index=prices_full.index, columns=[p[0], p[1]])
                    pair_weight_matrices[p].loc[current_date, p[0]] = w[p[0]]
                    pair_weight_matrices[p].loc[current_date, p[1]] = w[p[1]]
                    prev_w0 = 0.0 if pd.isna(master_weights.at[current_date, p[0]]) else master_weights.at[current_date, p[0]]
                    prev_w1 = 0.0 if pd.isna(master_weights.at[current_date, p[1]]) else master_weights.at[current_date, p[1]]
                    master_weights.loc[current_date, p[0]] = prev_w0 + w[p[0]]
                    master_weights.loc[current_date, p[1]] = prev_w1 + w[p[1]]

                avg_inno_z = np.mean([abs(mem['Entry_Z']) for mem in active_pairs.values()]) if active_pairs else 0.0
                avg_conviction = live_df["Conviction_Score"].mean() if not live_df.empty else 0.0
                
                metrics_history.append({"Date": current_date, "Ghost_Size": len(ghost_pool) if 'ghost_pool' in locals() else 0, "Promoted_Size": len(promoted_ghosts), "Live_Size": len(active_pairs), "Opt_Exp_Variance": opt.get("Portfolio_Variance", 0.0), "Effective_Gross": opt.get("Effective_Gross", 0.0), "Mean_Innovation_Z": avg_inno_z, "Mean_Conviction": avg_conviction, "Vol_Ratio": vol_ratio, "rebalanced": 1})

                print(f"  --> Pipeline Funnel : Ghost={len(ghost_pool) if 'ghost_pool' in locals() else 0} | Promoted={len(promoted_ghosts)} | Live={len(active_pairs)}", flush=True)
                if active_pairs:
                    print(f"  --> SLSQP Metrics   : Eff Gross={opt.get('Effective_Gross', 0):.2f} | Port Vol={np.sqrt(opt.get('Portfolio_Variance', 0)):.4f} | Turnover Drag={opt.get('Turnover_Cost', 0):.6f}", flush=True)
                    print(f"  --> Top Allocations :", flush=True)
                    sorted_allocs = sorted(active_pairs.keys(), key=lambda x: abs(allocs[x]['Sleeve_Weight']), reverse=True)[:3]
                    for p in sorted_allocs:
                        w_data = allocs[p]
                        mem = active_pairs[p]
                        print(f"      {p[0]}/{p[1]} | W_A: {w_data[p[0]]:.1%} | W_B: {w_data[p[1]]:.1%} | Inno Z (e_t): {mem['Entry_Z']:.2f} | GBDT Conviction: {mem['Conviction_Score']:.2f}", flush=True)
                print("-" * 60, flush=True)

                StatePersistenceEngine.save_checkpoint(params["state_persistence_file"], {
                    "active_pairs": active_pairs,
                    "cooldown_tracker": cooldown_tracker
                })

    # ==============================================================================
    # EXHAUSTIVE BACKTEST & FUNNEL ATTRIBUTION LOGIC
    # ==============================================================================
    cap = params["max_gross_exposure"]
    for df_w in [master_weights, ghost_weights, promoted_weights]:
        abs_sum = df_w.abs().sum(axis=1)
        exceed_mask = abs_sum > cap
        if exceed_mask.any(): 
            df_w.loc[exceed_mask] = df_w.loc[exceed_mask].div(abs_sum[exceed_mask], axis=0) * cap

    exec_weights = master_weights.shift(1).iloc[start_idx:]
    exec_ghost_weights = ghost_weights.shift(1).iloc[start_idx:]
    exec_promoted_weights = promoted_weights.shift(1).iloc[start_idx:]

    for p in pair_weight_matrices: 
        pair_weight_matrices[p] = pair_weight_matrices[p].ffill().fillna(0.0).shift(1)

    rebalance_metrics_df = pd.DataFrame(metrics_history).set_index("Date")

    exec_close_prices = prices_full.iloc[start_idx:]
    exec_open_prices = open_full.iloc[start_idx:]

    metrics_df = pd.DataFrame(index=exec_close_prices.index)
    metrics_df = metrics_df.join(rebalance_metrics_df, how="left")
    metrics_df["rebalanced"] = metrics_df["rebalanced"].fillna(0).astype(int)

    ffill_cols = ["Ghost_Size", "Promoted_Size", "Live_Size", "Opt_Exp_Variance", "Effective_Gross", "Mean_Innovation_Z", "Vol_Ratio", "Mean_Conviction"]
    metrics_df[ffill_cols] = metrics_df[ffill_cols].ffill().fillna(0.0)

    global_engine = VectorbtBacktestEngine(params)
    net_portfolio_live, val_live, true_gross_exp_series = global_engine.run_backtest(exec_close_prices, exec_open_prices, exec_weights)
    _, val_ghost, _ = global_engine.run_backtest(exec_close_prices, exec_open_prices, exec_ghost_weights)
    _, val_promoted, _ = global_engine.run_backtest(exec_close_prices, exec_open_prices, exec_promoted_weights)

    ret_live = val_live.pct_change().fillna(0.0)
    ret_ghost = val_ghost.pct_change().fillna(0.0)
    ret_promoted = val_promoted.pct_change().fillna(0.0)

    metrics_df["Live_Realized_Return"] = ret_live
    metrics_df["Ghost_Realized_Return"] = ret_ghost
    metrics_df["Promoted_Realized_Return"] = ret_promoted

    n_days = max(1, len(ret_live))
    ann_factor = 252.0 / n_days

    tot_ret_ghost = (val_ghost.iloc[-1] / params["initial_capital"]) - 1.0
    tot_ret_promoted = (val_promoted.iloc[-1] / params["initial_capital"]) - 1.0
    tot_ret_live = (val_live.iloc[-1] / params["initial_capital"]) - 1.0

    ann_ret_ghost = (((1.0 + tot_ret_ghost) ** ann_factor) - 1.0) * 100.0
    ann_ret_promoted = (((1.0 + tot_ret_promoted) ** ann_factor) - 1.0) * 100.0
    ann_ret_live = (((1.0 + tot_ret_live) ** ann_factor) - 1.0) * 100.0

    sortino_ghost = compute_sortino_ratio(ret_ghost)
    sortino_promoted = compute_sortino_ratio(ret_promoted)
    sortino_live = compute_sortino_ratio(ret_live)

    ret_delta_gbdt = ann_ret_promoted - ann_ret_ghost
    ret_delta_optimizer = ann_ret_live - ann_ret_promoted

    sortino_delta_gbdt = sortino_promoted - sortino_ghost
    sortino_delta_optimizer = sortino_live - sortino_promoted

    benchmark_returns = market_ret.iloc[start_idx:]
    neutrality_engine = MarketNeutralityDiagnosticEngine(max_beta=0.03, max_r2=0.01)
    neutrality_metrics_df, verdict = neutrality_engine.analyze(strategy_returns=ret_live, benchmark_returns=benchmark_returns)

    tearsheet_engine = CommitteeTearsheetEngine(params)
    committee_report = tearsheet_engine.generate_tearsheets(prices_full, pair_weight_matrices)

    stats_df = net_portfolio_live.stats()
    stats_df["Max Gross Exposure [%]"] = float(true_gross_exp_series.max() * 100.0)

    order_gen = LiveMOOOrderGenerator(min_order_usd=500.0, max_adv_part=params.get("max_adv_participation", 0.015))
    rolling_adv_shares = (prices_full.iloc[-20:].mean() * 50_000).fillna(1_000_000) 
    final_target_weights = master_weights.ffill().fillna(0.0).iloc[-1]
    
    moo_blotter = order_gen.generate_blotter(
        target_weights=final_target_weights,
        current_positions={},
        latest_prices=prices_full.iloc[-1],
        rolling_adv_shares=rolling_adv_shares,
        portfolio_nav=params["initial_capital"]
    )

    summary_data = {
        "stats_df": stats_df,
        "ann_ret_ghost": ann_ret_ghost,
        "sortino_ghost": sortino_ghost,
        "ann_ret_promoted": ann_ret_promoted,
        "sortino_promoted": sortino_promoted,
        "ann_ret_live": ann_ret_live,
        "sortino_live": sortino_live,
        "ret_delta_gbdt": ret_delta_gbdt,
        "sortino_delta_gbdt": sortino_delta_gbdt,
        "ret_delta_optimizer": ret_delta_optimizer,
        "sortino_delta_optimizer": sortino_delta_optimizer,
        "verdict": verdict,
        "neutrality_metrics_df": neutrality_metrics_df
    }

    return committee_report, net_portfolio_live, metrics_df, moo_blotter, summary_data

# ==============================================================================
# 9. DIAGNOSTIC TEARSHEET PLOTTER & REPORTING ENGINE
# ==============================================================================
def _get_next_run_dir(base_dir: str = "results") -> str:
    os.makedirs(base_dir, exist_ok=True)
    existing_ids = [int(d) for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d)) and d.isdigit()]
    run_id = max(existing_ids) + 1 if existing_ids else 1
    run_dir = os.path.join(base_dir, str(run_id))
    os.makedirs(run_dir, exist_ok=True)
    return run_dir

def format_independent_axis(ax, rebalance_dates):
    for r_date in rebalance_dates:
        ax.axvline(x=r_date, color="red", linestyle=":", alpha=0.4, linewidth=1.2)
    ax.grid(True, linestyle=":", alpha=0.6)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

def print_execution_summary(
    metrics_df: pd.DataFrame, 
    stats_df: pd.DataFrame, 
    committee_report: pd.DataFrame, 
    moo_blotter: pd.DataFrame,
    ann_ret_ghost: float, sortino_ghost: float,
    ann_ret_promoted: float, sortino_promoted: float,
    ann_ret_live: float, sortino_live: float,
    ret_delta_gbdt: float, sortino_delta_gbdt: float,
    ret_delta_optimizer: float, sortino_delta_optimizer: float,
    verdict: dict, neutrality_metrics_df: pd.DataFrame
):
    print("\n" + "=" * 85, flush=True)
    print("FUNNEL CONVERSION ATTRIBUTION & MARGINAL PERFORMANCE REPORT", flush=True)
    print("=" * 85, flush=True)
    print(f"  {'Funnel Tier / Portfolio Stage':<38} | {'Ann. Return [%]':<18} | {'Sortino Ratio':<18}", flush=True)
    print("-" * 85, flush=True)
    print(f"  {'Ghost Portfolio (Raw Universe)':<38} | {ann_ret_ghost:17.2f}% | {sortino_ghost:18.4f}", flush=True)
    print(f"  {'Promoted Portfolio (GBDT Soft-Gated)':<38} | {ann_ret_promoted:17.2f}% | {sortino_promoted:18.4f}", flush=True)
    print(f"  {'Live Portfolio (SLSQP Optimized)':<38} | {ann_ret_live:17.2f}% | {sortino_live:18.4f}", flush=True)
    print("-" * 85, flush=True)
    print(f"  {'--> GBDT Selection Delta':<38} | {ret_delta_gbdt:+17.2f}% | {sortino_delta_gbdt:+18.4f}", flush=True)
    print(f"  {'--> Optimizer Allocation Delta':<38} | {ret_delta_optimizer:+17.2f}% | {sortino_delta_optimizer:+18.4f}", flush=True)
    print("=" * 85, flush=True)

    print("\n" + "=" * 85, flush=True)
    print("QUANTITATIVE MARKET NEUTRALITY & FACTOR DIAGNOSTIC AUDIT", flush=True)
    print("=" * 85, flush=True)
    print(neutrality_metrics_df.to_string(index=False), flush=True)
    print("-" * 85, flush=True)
    print(f"  VERDICT: Market Neutral = {verdict['Market_Neutral']} | Failures: {verdict['Core_Failure_Reason']}", flush=True)
    print("=" * 85, flush=True)

    print("\n" + "=" * 85, flush=True)
    if not committee_report.empty:
        print("INVESTMENT COMMITTEE PAIR REVIEW TEARSHEET", flush=True)
        print("=" * 85, flush=True)
        print(committee_report.to_string(index=False), flush=True)
    else:
        print("INVESTMENT COMMITTEE PAIR REVIEW TEARSHEET: NO TRADES COMPLETED.", flush=True)

    print("\n" + "=" * 85, flush=True)
    print("WALK-FORWARD NET PORTFOLIO STATS (POST-FEES & EXECUTION LAG)", flush=True)
    print("=" * 85, flush=True)
    print(stats_df.to_string(), flush=True)

    print("\n" + "=" * 85, flush=True)
    print("ACTIONABLE MARKET-ON-OPEN (MOO) ORDER BLOTTER FOR NEXT SESSION", flush=True)
    print("=" * 85, flush=True)
    print(moo_blotter.to_string(index=False), flush=True)

def plot_diagnostic_tearsheet(
    metrics_df: pd.DataFrame,
    initial_capital: float = 250_000.0,
    committee_report: Optional[pd.DataFrame] = None,
    net_portfolio: Optional[vbt.Portfolio] = None,
    moo_blotter: Optional[pd.DataFrame] = None,
    save: bool = True,
    prefix: str = "dev_"
):
    if metrics_df.empty: return
    df = metrics_df.copy()
    if not isinstance(df.index, pd.DatetimeIndex): df.index = pd.to_datetime(df.index)
    rebalance_dates = df[df["rebalanced"] == 1].index if "rebalanced" in df.columns else []

    output_dir = None
    if save:
        output_dir = _get_next_run_dir("results")
        print(f"\n[Export Engine] Saving artifacts to: '{output_dir}/'", flush=True)
        metrics_df.to_csv(os.path.join(output_dir, f"{prefix}metrics.csv"))
        if committee_report is not None and not committee_report.empty: committee_report.to_csv(os.path.join(output_dir, f"{prefix}committee_report.csv"), index=False)
        if moo_blotter is not None: moo_blotter.to_csv(os.path.join(output_dir, f"{prefix}moo_blotter.csv"), index=False)
        if net_portfolio is not None:
            try: net_portfolio.trades.records_readable.to_csv(os.path.join(output_dir, f"{prefix}trade_records.csv"), index=False)
            except Exception: pass
            try: net_portfolio.stats().to_csv(os.path.join(output_dir, f"{prefix}portfolio_stats.csv"))
            except Exception: pass

    fig1, ax1 = plt.subplots(figsize=(16, 4))
    ax1.plot(df.index, df["Ghost_Size"], label="Ghost Universe", color="#2b5c8f", linestyle="--", linewidth=1.5)
    ax1.plot(df.index, df["Promoted_Size"], label="Promoted Universe (GBDT Gated)", color="#e07a5f", linestyle="-.", linewidth=1.5)
    ax1.step(df.index, df["Live_Size"], label="Live Portfolio", color="#2a9d8f", where="post", linewidth=2.5)
    ax1.set_title("Universe Funnel Sizing (Pair Counts)", fontsize=11, fontweight="bold", loc="left")
    ax1.legend(loc="upper left", bbox_to_anchor=(1.02, 1), borderaxespad=0., frameon=True)
    format_independent_axis(ax1, rebalance_dates)
    plt.tight_layout()
    if save and output_dir: fig1.savefig(os.path.join(output_dir, f"{prefix}funnel_sizing.png"), dpi=300, bbox_inches="tight")
    plt.show()

    fig3, ax3 = plt.subplots(figsize=(16, 4))
    ax3.plot(df.index, df["Mean_Innovation_Z"], label="Mean |Innovation Z| (|e_t|)", color="#e76f51", linewidth=2.0)
    ax3.set_title("Kalman Innovation Z-Score (|e_t|)", fontsize=11, fontweight="bold", loc="left")
    ax3.legend(loc="upper left", bbox_to_anchor=(1.08, 1), borderaxespad=0., frameon=True)
    format_independent_axis(ax3, rebalance_dates)
    plt.tight_layout()
    if save and output_dir: fig3.savefig(os.path.join(output_dir, f"{prefix}kalman_z.png"), dpi=300, bbox_inches="tight")
    plt.show()

    fig4, ax4 = plt.subplots(figsize=(16, 4))
    ax4.plot(df.index, df["Mean_Conviction"], label="Mean GBDT Conviction (s_i)", color="#457b9d", linewidth=2.0)
    ax4.axhline(0.40, color="black", linewidth=0.8, linestyle="--", label="Soft-Gate Hurdle (0.40)")
    ax4.set_title("Adaptive HistGBDT Conviction Trajectory", fontsize=11, fontweight="bold", loc="left")
    ax4.legend(loc="upper left", bbox_to_anchor=(1.02, 1), borderaxespad=0., frameon=True)
    format_independent_axis(ax4, rebalance_dates)
    plt.tight_layout()
    if save and output_dir: fig4.savefig(os.path.join(output_dir, f"{prefix}gbdt_conviction.png"), dpi=300, bbox_inches="tight")
    plt.show()

    fig5, ax5 = plt.subplots(figsize=(16, 5))
    live_cum = (1.0 + df.get("Live_Realized_Return", pd.Series(0.0, index=df.index)).fillna(0.0)).cumprod() * initial_capital
    promoted_cum = (1.0 + df.get("Promoted_Realized_Return", pd.Series(0.0, index=df.index)).fillna(0.0)).cumprod() * initial_capital
    ghost_cum = (1.0 + df.get("Ghost_Realized_Return", pd.Series(0.0, index=df.index)).fillna(0.0)).cumprod() * initial_capital

    ax5.plot(df.index, live_cum, label="Live Realized Value ($)", color="#2a9d8f", linewidth=2.5)
    ax5.plot(df.index, promoted_cum, label="Promoted Shadow Value ($)", color="#e07a5f", linestyle="-.", linewidth=1.5)
    ax5.plot(df.index, ghost_cum, label="Ghost Shadow Value ($)", color="#2b5c8f", linestyle="--", linewidth=1.5)
    
    ax5.axhline(initial_capital, color="black", linewidth=0.8)
    ax5.set_title("Cumulative Performance Horizon & Funnel Attribution ($)", fontsize=11, fontweight="bold", loc="left")
    ax5.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda x, loc: "{:,}".format(int(x))))
    ax5.legend(loc="upper left", bbox_to_anchor=(1.02, 1), borderaxespad=0., frameon=True)
    format_independent_axis(ax5, rebalance_dates)
    plt.tight_layout()
    if save and output_dir: fig5.savefig(os.path.join(output_dir, f"{prefix}cumulative_performance.png"), dpi=300, bbox_inches="tight")
    plt.show()

# ==============================================================================
# MAIN EXECUTION ENTRY POINT
# ==============================================================================
if __name__ == "__main__":
    committee_report, net_portfolio, metrics, moo_blotter, summary_data = run_unified_pipeline(params=run_params)
    
    if metrics is not None and not metrics.empty:
        print_execution_summary(
            metrics_df=metrics,
            committee_report=committee_report,
            moo_blotter=moo_blotter,
            **summary_data
        )
        
        plot_diagnostic_tearsheet(
            metrics_df=metrics,
            initial_capital=run_params.get("initial_capital", 250_000.0),
            committee_report=committee_report,
            net_portfolio=net_portfolio,
            moo_blotter=moo_blotter,
            save=True,
            prefix="dev_"
        )

STARTING ROBUST 2-STAGE STAT-ARB ENGINE (TRUNCATED GHOST -> HIST-GBDT SLSQP)

[Screening Pass @ 2024-08-28 | Progress:   0.2% (1/500)] SciPy C-Accelerated Search...

[Dynamic Rebalance @ 2024-08-28 | Progress:   0.2% (1/500)] Cause: INITIAL_ENTRY | Vol Ratio: 0.85 | VIX: 11.5
  --> Pipeline Funnel : Ghost=45 | Promoted=30 | Live=15
  --> SLSQP Metrics   : Eff Gross=1.79 | Port Vol=0.0757 | Turnover Drag=0.001435
  --> Top Allocations :
      ITW/SNA | W_A: -18.8% | W_B: 14.9% | Inno Z (e_t): 0.64 | GBDT Conviction: 0.50
      MRSH/MCO | W_A: -18.8% | W_B: 12.3% | Inno Z (e_t): 0.32 | GBDT Conviction: 0.50
      GOOG/GOOGL | W_A: 18.8% | W_B: -18.6% | Inno Z (e_t): -0.04 | GBDT Conviction: 0.50
------------------------------------------------------------

[Dynamic Rebalance @ 2024-08-29 | Progress:   0.4% (2/500)] Cause: SIGNAL_EXIT | Vol Ratio: 0.87 | VIX: 11.5
  --> Pipeline Funnel : Ghost=33 | Promoted=30 | Live=21
  --> SLSQP Metrics   : Eff Gross=1.79 | Port Vol=0.0441 | Turnover D